# This is a signal transformation code

In [ ]:
import numpy as np
from scipy.interpolate import interp1d
from scipy.fftpack import fft, ifft, fftshift

## Time Domain Transformation

In [ ]:
def forcast_mask(src_sig,N,fs=125) :

    masked_sig = np.copy(src_sig)
    masked_sig[-int(N*fs):,:] = 0

    return masked_sig

In [ ]:
def slise_mask(src_sig,fs=125) :
    
    w1 = np.random.randint(low=fs,high=int(4.5*fs))
    w2 = np.random.randint(low=w1+fs,high=6*fs)
    w3 = np.random.randint(low=w2+fs,high=7*fs)
    w4 = np.random.randint(low=w3+fs,high=8*fs)

    masked_sig = np.copy(src_sig)
    masked_sig[w1:int(w1+fs/2),:] = 0
    masked_sig[w2:int(w2+fs/2),:] = 0
    masked_sig[w3:int(w3+fs/2),:] = 0
    masked_sig[w4:int(w4+fs/2),:] = 0

    return masked_sig

In [ ]:
def gauss_noisy(src_sig,snr=10) :

    noise = np.random.randn(*src_sig.shape)
    noise = noise - np.mean(noise)
    signal_power = np.linalg.norm(src_sig - np.mean(src_sig))** 2 / src_sig.size
    sigma = signal_power / np.power(10,(snr/10))
    noise = (np.sqrt(sigma) / np.std(noise)) * noise

    noisy_sig = noise + src_sig

    return noisy_sig

In [ ]:
def amp_invers(src_sig,fs=125) :

    inver_sig = np.copy(src_sig)
    w1 = np.random.randint(low=2*fs,high=int(5.5*fs))
    inver_sig[w1:w1+3*fs,:] = 1-src_sig[w1:w1+3*fs,:]

    return inver_sig

In [ ]:
def time_invers(src_sig,fs=125) :

    inver_sig = np.copy(src_sig)
    w1 = np.random.randint(low=2*fs,high=int(5.5*fs))
    inver_sig[w1:w1+3*fs,:] = src_sig[w1+3*fs:w1:-1,:]

    return inver_sig

In [ ]:
def time_warping(src_sig,fs=125) :

    src_t = np.linspace(start=0,stop=3,num=3*fs,endpoint=True)
    tgt_ts = np.linspace(start=0,stop=3,num=2*fs,endpoint=True)
    tgt_tl = np.linspace(start=0,stop=3,num=4*fs,endpoint=True)
    
    flg = np.random.randint(low=2)

    if flg :
        fdwn = interp1d(x=src_t,y=src_sig[6*fs:9*fs,0])
        fups = interp1d(x=src_t,y=src_sig[1*fs:4*fs,0])

        tgt_ss = fdwn(tgt_ts)
        tgt_sl = fups(tgt_tl)

        twrp_sig = np.append(np.append(np.append(np.append(src_sig[:fs,0],tgt_sl),src_sig[4*fs:6*fs,0]),tgt_ss),src_sig[9*fs:,0])
    else :
        fdwn = interp1d(x=src_t,y=src_sig[1*fs:4*fs,0])
        fups = interp1d(x=src_t,y=src_sig[6*fs:9*fs,0])

        tgt_ss = fdwn(tgt_ts)
        tgt_sl = fups(tgt_tl)

        twrp_sig = np.append(np.append(np.append(np.append(src_sig[:fs,0],tgt_ss),src_sig[4*fs:6*fs,0]),tgt_sl),src_sig[9*fs:,0])

    return twrp_sig[:,np.newaxis]

## Frequency Domain Transformation

In [ ]:
def frq_forcast_mask(src_sig,fs=125) :

    f_sig = fft(src_sig[:,0])
    f_sig[35:1250-35] = 0
    masked_sig = np.real(ifft(f_sig))

    masked_sig = (masked_sig-np.min(masked_sig))/(np.max(masked_sig)-np.min(masked_sig))
    
    return masked_sig[:,np.newaxis]

In [ ]:
def frq_slise_mask(src_sig,fs=125) :

    f_sig = fft(src_sig[:,0])
    w1 = np.random.randint(low=5,high=int(5+20))
    w2 = np.random.randint(low=60,high=60+125)
    w3 = np.random.randint(low=1250-185,high=1250-60)
    w4 = np.random.randint(low=1250-55,high=1250-30)
    
    f_sig[w1:int(w1+25)] = 0
    f_sig[w2:int(w2+fs/2)] = 0
    f_sig[w3:int(w3+fs/2)] = 0
    f_sig[w4:int(w4+25)] = 0    
    
    masked_sig = np.real(ifft(f_sig))

    masked_sig = (masked_sig-np.min(masked_sig))/(np.max(masked_sig)-np.min(masked_sig))
    
    return masked_sig[:,np.newaxis]

In [ ]:
def frq_shift(src_sig) :

    f_sig = fft(src_sig[:,0])
    frq_shift = fftshift(f_sig)
    shift_sig = np.real(ifft(frq_shift))

    shift_sig = (shift_sig-np.min(shift_sig))/(np.max(shift_sig)-np.min(shift_sig))
    
    return shift_sig[:,np.newaxis]

In [ ]:
def frq_am(src_sig) :
    t = np.linspace(start=0,stop=10,num=1250)
    am = 0.5*(1+1.5*src_sig[:,0])*np.sin(2*3.1415*20*t)

    am = (am-np.min(am))/(np.max(am)-np.min(am))

    return am[:,np.newaxis]

In [ ]:
def frq_fm(src_sig) :
    t = np.linspace(start=0,stop=10,num=1250)
    fm = np.sin(2*3.1415*(20)*(t) + 0.75*np.cumsum(src_sig[:,0]))

    fm = (fm-np.min(fm))/(np.max(fm)-np.min(fm))
    
    return fm[:,np.newaxis]

In [ ]:
def frq_warpping(src_sig) :
    
    f_sig = fft(src_sig[:,0])
    frq_shift = fftshift(f_sig)
    src_f = np.linspace(start=1,stop=250,num=250,endpoint=True)
    tgt_f = np.linspace(start=1,stop=250,num=1250,endpoint=True)
    src_frq = frq_shift[500:750]
    intp_func = interp1d(x=src_f,y=src_frq)
    wr_frq = intp_func(tgt_f)
    tt_wr_f = wr_frq + fftshift(wr_frq)
    tt_wr_f[0] = f_sig[0]/2
    frq_wr_sig = np.real(ifft(tt_wr_f/(625)))
    
    frq_wr_sig = (frq_wr_sig-np.min(frq_wr_sig))/(np.max(frq_wr_sig)-np.min(frq_wr_sig))
    
    return frq_wr_sig[:,np.newaxis]
